<a href="https://colab.research.google.com/github/vipra2703/NNDL_LAB/blob/main/Experiment6/LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------------- hyperparameters ----------------
batch_size    = 64      # sequences per batch
block_size    = 128     # context length (characters)
n_embd        = 256     # embedding size
hidden_size   = 512     # LSTM hidden size
n_layer       = 2       # stacked LSTM layers
dropout       = 0.2
learning_rate = 2e-3
max_iters     = 5000
eval_interval = 500
eval_iters    = 100
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)

# ---------------- data (read from website) ----------------
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
with urllib.request.urlopen(url) as response:
    text = response.read().decode('utf-8')
print(f"Loaded {len(text):,} characters from website")

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]

data = torch.tensor(encode(text), dtype=torch.long)
n1 = int(0.8 * len(data))
n2 = int(0.9 * len(data))
train_data = data[:n1]        # 80% train
val_data   = data[n1:n2]      # 10% validation
# data[n2:] is the test set -> used only in test.py

def get_batch(split):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# ---------------- model ----------------
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, hidden_size, n_layer, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, n_embd)
        self.lstm = nn.LSTM(input_size=n_embd, hidden_size=hidden_size,
                            num_layers=n_layer, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.lm_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, idx, targets=None, hidden=None):
        x = self.embedding(idx)
        out, hidden = self.lstm(x, hidden)
        logits = self.lm_head(self.dropout(out))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss, hidden

model = LSTMLanguageModel(vocab_size, n_embd, hidden_size, n_layer, dropout).to(device)
print(f'{sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters | device: {device}')

# ---------------- evaluation helper ----------------
@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss, _ = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# ---------------- training loop ----------------
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
best_val = float('inf')

for it in range(max_iters):
    if it % eval_interval == 0 or it == max_iters - 1:
        l = estimate_loss()
        print(f"step {it}: train loss {l['train']:.4f}, val loss {l['val']:.4f}")

        # save the model whenever validation loss improves
        if l['val'] < best_val:
            best_val = l['val']
            torch.save({
                'model_state': model.state_dict(),
                'chars': chars,
                'config': {
                    'n_embd': n_embd, 'hidden_size': hidden_size,
                    'n_layer': n_layer, 'dropout': dropout,
                    'block_size': block_size,
                },
            }, 'lstm_model.pt')
            print(f"  -> saved best model (val loss {best_val:.4f})")

    xb, yb = get_batch('train')
    _, loss, _ = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

print(f"Training done. Best val loss: {best_val:.4f}. Model saved to lstm_model.pt")

Loaded 1,115,394 characters from website
3.73M parameters | device: cuda
step 0: train loss 4.1679, val loss 4.1683
  -> saved best model (val loss 4.1683)
step 500: train loss 1.2860, val loss 1.4626
  -> saved best model (val loss 1.4626)
step 1000: train loss 1.1672, val loss 1.4344
  -> saved best model (val loss 1.4344)
step 1500: train loss 1.0857, val loss 1.4431
step 2000: train loss 1.0217, val loss 1.4681
step 2500: train loss 0.9699, val loss 1.4885
step 3000: train loss 0.9340, val loss 1.5361
step 3500: train loss 0.8985, val loss 1.5526
step 4000: train loss 0.8759, val loss 1.5610
step 4500: train loss 0.8496, val loss 1.5778
step 4999: train loss 0.8316, val loss 1.5938
Training done. Best val loss: 1.4344. Model saved to lstm_model.pt


In [3]:
import math
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(1337)

# ---------------- model (same definition as train.py) ----------------
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, hidden_size, n_layer, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, n_embd)
        self.lstm = nn.LSTM(input_size=n_embd, hidden_size=hidden_size,
                            num_layers=n_layer, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.lm_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, idx, targets=None, hidden=None):
        x = self.embedding(idx)
        out, hidden = self.lstm(x, hidden)
        logits = self.lm_head(self.dropout(out))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss, hidden

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8):
        logits, _, hidden = self(idx)
        for _ in range(max_new_tokens):
            probs = F.softmax(logits[:, -1, :] / temperature, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
            logits, _, hidden = self(next_id, hidden=hidden)
        return idx

# ---------------- load trained model ----------------
ckpt = torch.load('lstm_model.pt', map_location=device, weights_only=True)
chars = ckpt['chars']
cfg = ckpt['config']
block_size = cfg['block_size']

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

model = LSTMLanguageModel(len(chars), cfg['n_embd'], cfg['hidden_size'],
                          cfg['n_layer'], cfg['dropout']).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()                       # dropout off for testing
print("Model loaded from lstm_model.pt")

# ---------------- test data (read from website, same split as train.py) ----------------
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
with urllib.request.urlopen(url) as response:
    text = response.read().decode('utf-8')

data = torch.tensor(encode(text), dtype=torch.long)
n2 = int(0.9 * len(data))
test_data = data[n2:]              # last 10% - never seen in training
print(f"Test set: {len(test_data):,} characters")

# ---------------- evaluate ----------------
@torch.no_grad()
def evaluate(d):
    total_loss, total_correct, total_chars = 0.0, 0, 0
    for i in range(0, len(d) - block_size - 1, block_size):
        x = d[i:i+block_size].unsqueeze(0).to(device)
        y = d[i+1:i+block_size+1].unsqueeze(0).to(device)
        logits, loss, _ = model(x, y)
        total_loss += loss.item() * block_size
        total_correct += (logits.argmax(dim=-1) == y).sum().item()
        total_chars += block_size
    avg_loss = total_loss / total_chars
    return avg_loss, math.exp(avg_loss), 100 * total_correct / total_chars

test_loss, test_ppl, test_acc = evaluate(test_data)
print(f"Test loss       : {test_loss:.4f}")
print(f"Test perplexity : {test_ppl:.2f}")
print(f"Test accuracy   : {test_acc:.2f}%")

# ---------------- generate sample text ----------------
prompt = "ROMEO:\n"
context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
print("\n--- Generated text ---")
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))

Model loaded from lstm_model.pt
Test set: 111,540 characters
Test loss       : 1.6605
Test perplexity : 5.26
Test accuracy   : 52.90%

--- Generated text ---
ROMEO:
And they this swear to look to be a pack.
And take Oxford that are barriage out of his back:
And all away with himself that makes
Whereof it is confirm'd like earth,
Then come all maids, by Saint Walthough I was speak:
The wanton she could you can never let it love.
I cannot beard whose laws to thee
And thou what evil sound more rivers will not pass
The foreshes and thrust for a common must.

BAGOT:
I thank your grace? and thou art too discovers.
We are help to hear a married in his charge!
And,
